In [1]:
from datasets import load_dataset

In [2]:
mnist = load_dataset("ylecun/mnist")
mnist

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 60000
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 10000
    })
})

In [3]:
from torchvision import transforms

In [4]:
def mnist_to_tensor(samples):
    t = transforms.ToTensor()
    samples["image"] = [t(image) for image in samples["image"]]
    return samples

In [5]:
mnist = mnist.with_transform(mnist_to_tensor)
mnist["train"] = mnist["train"].shuffle(seed=1337)

In [6]:
x = mnist["train"]["image"][0]
x.min(), x.max()

(tensor(0.), tensor(1.))

In [7]:
from torch.utils.data import DataLoader

In [8]:
bs=64
train_dataloader = DataLoader(mnist["train"]["image"],batch_size=bs)

In [9]:
from torch import nn

def conv_block(in_channels, out_channels, kernel_size=4, stride=2, padding=1):
    return nn.Sequential(
        nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding
        ),
        nn.BatchNorm2d(out_channels),
        nn.ReLU()
    )
    
class VAEEncoder(nn.Module):
    def __init__(self, in_channels, latent_dims):
        super().init()

        self.conv_layers = nn.Sequential(
            conv_block(in_channels,128),
            conv_block(128,256),
            conv_block(256,512),
            conv_block(512,1024)
        )

        #Define fully connected layers for mean and log-variance
        self.mu = nn.Linear(1024,latent_dims)
        self.logvar = nn.Linear(1024,latent_dims)

    def forward(self,x):
        bs = x.shape[0]
        x = self.conv_layers(x)
        x = x.reshape(bs, -1)
        mu = self.mu(x)
        logvar = self.logvar(x)
        return(mu, logvar)

In [11]:
def conv_transpose_block(
    in_channels,
    out_channels,
    kernel_size=3,
    stride=2,
    padding=1,
    output_padding=0,
    with_act = True
):
    modules = [
        nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            output_padding = output_padding
        )
    ]
    if with_act:
        modules.append(nn.BatchNorm2d(out_channels))
        modules.append(nn.ReLU())
    return nn.Sequential(*modules)

class Decoder(nn.Module):
    def __init__(self, out_channels):
        super().__init__()

        self.linear = nn.Linear(16,1024*4*4)
        self.t_conv1 = conv_transpose_block(1024,512)
        self.t_conv2 = conv_transpose_block(512,256,output_padding=1)
        self.t_conv3 = conv_transpose_block(256,out_channels, output_padding=1)

    def forward(self,x):
        bs = x.shape[0]
        x= self.linear(x) #(bs, 1024*4*4)
        x= x.reshape((bs,1024,4,4)) #(bs,1024,4,4)
        x= self.t_conv1(x) #(bs, 512, 7, 7)
        x= self.t_conv2(x) #(bs, 256, 14, 14)
        x= self.t_conv3(x) #(bs, 1, 28, 28)
        return x

In [12]:
class VAE(nn.Module):
    def __init__(self, in_channels, latent_dims):
        super().__init__()

        self.encoder = VAEEncoder(in_channels, latent_dims)
        self.decoder = Decoder(in_channels, latent_dims)

        def encode(self,x):
            #returns mu, log_var
            return self.encoder(x)

        def decode(self,x):
            return self.decoder(z)

        def forward(self,x):
            #obtain parameters of the gaussian distrubition
            mu, logvar = self.encode(x)

            #sample from the distrubition
            std = torch.exp(0.5*logvar)
            z = self.sample(mu,std)

            #decode the latent point to pixel space
            reconstructed = self.decode(z)

            #return the reconstructed image, and also the mu and logvar
            #so we can compute a distrubition loss
            return reconstructed,mu,logvar
            
        def sample(self,mu,std):
            #Reparametrization trick
            #Sample from N(0,I), translate and scale
            eps = torch.randn_like(std)
            return mu + eps * std